# Black CEO Portfolio: Visual Weight Transition Map Prototype
**Sub-issue #186:** Prototype visual weight transition maps across historical rebalancing regimes (`2000–2026`).

This notebook inspects point-in-time constituent weight trajectories generated by `src/optimizer.py` and `src/backtest.py`, verifying:
1. **Sparse-Universe Cash-Drag Defense ($N_t < 3$):** Equity exposure scales proportionally ($B_t = N_t / 3$) with residual capital held in risk-free cash.
2. **Dynamic Concentration Ceiling:** No single constituent exceeds $w_{\max}(N_t) = \min(0.35, 2/N_t)$ on any rebalance date.
3. **Regime Transition Dynamics:** Smooth multi-decade rotation as Black CEO tenures enter and exit the S&P / institutional roster.

## 1. Environment Setup & Data Ingestion
Load the 32-ticker daily price matrix (`data/prices_daily.csv`) and the $t+1$ execution-lagged universe mask (`data/universe_mask.csv`).

In [ ]:
import sys
from pathlib import Path

# Ensure repository root is on sys.path so `src.*` imports resolve cleanly
REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebook" else Path.cwd().resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from src.backtest import (
    PRICES_PATH,
    MASK_PATH,
    compute_returns,
    compute_target_weights,
    get_rebalance_mask,
)

# Load aligned market prices and point-in-time active constituent mask
prices_df = pd.read_csv(PRICES_PATH, index_col=0, parse_dates=True)
mask_df = pd.read_csv(MASK_PATH, index_col=0, parse_dates=True).astype(bool)
returns_df = compute_returns(prices_df)

print(f"Loaded prices shape: {prices_df.shape} | Universe mask shape: {mask_df.shape}")
print(f"Date range: {mask_df.index.min().date()} -> {mask_df.index.max().date()}")

## 2. Compute Target Weight Trajectories & Verify Concentration Bounds
Generate daily target weight matrices for both `equal_weight` and `convex_quality` using the synchronized monthly + event-driven rebalancing calendar (`get_rebalance_mask`).

In [ ]:
# Compute synchronized weight trajectories for Equal Weight and Convex Quality
ew_weights, ew_cash, n_t = compute_target_weights(
    universe_mask=mask_df,
    asset_returns=returns_df,
    allocation_scheme="equal_weight",
    rebalance_monthly=True,
)

cq_weights, cq_cash, _ = compute_target_weights(
    universe_mask=mask_df,
    asset_returns=returns_df,
    allocation_scheme="convex_quality",
    lambda_tilt=0.5,
    rebalance_monthly=True,
)

# Extract synchronized rebalance dates to inspect hard single-issuer concentration caps
rebal_flags = get_rebalance_mask(mask_df, rebalance_monthly=True)
max_rebal_weight_cq = float(cq_weights.loc[rebal_flags].max().max())

print(f"Total trading sessions: {len(mask_df)} | Synchronized rebalance events: {int(rebal_flags.sum())}")
print(f"Max single-asset weight on any Convex Quality rebalance date: {max_rebal_weight_cq:.4f} (Cap <= 0.3500)")

## 3. Stacked-Area Visual Weight Transition Map (`2000–2026`)
Visualize how portfolio capital transitions across top constituents and the defensive cash buffer (`Cash_Buffer`) over the full 26-year backtest horizon.

In [ ]:
# Resample to month-end snapshots for clean visual rendering of multi-decade transitions
cq_monthly = cq_weights.resample("ME").last()
cash_monthly = cq_cash.resample("ME").last()

# Isolate top 10 constituents by average allocation weight; group remaining active names into 'Other_Active'
mean_alloc = cq_monthly.mean().sort_values(ascending=False)
top_tickers = list(mean_alloc.head(10).index)
other_tickers = [c for c in cq_monthly.columns if c not in top_tickers]

plot_df = cq_monthly[top_tickers].copy()
plot_df["Other_Active"] = cq_monthly[other_tickers].sum(axis=1)
plot_df["Cash_Buffer"] = cash_monthly

fig, ax = plt.subplots(figsize=(14, 6))
colors = list(plt.cm.tab20.colors[:len(top_tickers)]) + ["#95a5a6", "#ecf0f1"]
ax.stackplot(plot_df.index, plot_df.T.values, labels=plot_df.columns, colors=colors, alpha=0.92)

# Overlay the 35% institutional single-issuer concentration ceiling
ax.axhline(0.35, color="#c0392b", linestyle="--", linewidth=1.3, label="35% Single-Asset Cap")

ax.set_title("Convex Quality Optimizer: Historical Portfolio Weight Transition Map (2000–2026)", fontsize=13, fontweight="bold")
ax.set_ylabel("Portfolio Allocation Weight")
ax.set_ylim(0.0, 1.0)
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0), frameon=True, fontsize=9)
ax.grid(axis="y", linestyle=":", alpha=0.5)

plt.tight_layout()
plt.show()

## 4. Constituent Weight Heatmap: Equal Weight vs. Convex Quality
Compare cross-sectional weight dispersion across all 32 tickers over the recent multi-constituent regime (`2020–2026`) to observe how Ledoit-Wolf variance minimization and quality tilts reweight active peers relative to uniform $1/N_t$ allocation.

In [ ]:
# Slice modern multi-constituent era (2020-2026) at quarter-end intervals for crisp heatmap inspection
ew_modern = ew_weights.loc["2020-01-01":].resample("QE").last()
cq_modern = cq_weights.loc["2020-01-01":].resample("QE").last()

# Filter to constituents active at least once during 2020-2026
active_cols = cq_modern.columns[(cq_modern > 0).any(axis=0)]
ew_sub = ew_modern[active_cols].T
cq_sub = cq_modern[active_cols].T

fig, axes = plt.subplots(1, 2, figsize=(16, 7), sharey=True)

im0 = axes[0].imshow(ew_sub.values, aspect="auto", cmap="Blues", vmin=0.0, vmax=0.35)
axes[0].set_title("Equal Weight Baseline (2020–2026)", fontsize=12, fontweight="bold")
axes[0].set_yticks(range(len(active_cols)))
axes[0].set_yticklabels(active_cols, fontsize=9)
axes[0].set_xticks(range(0, len(ew_sub.columns), 4))
axes[0].set_xticklabels([d.strftime("%Y") for d in ew_sub.columns[::4]], rotation=45)

im1 = axes[1].imshow(cq_sub.values, aspect="auto", cmap="YlGnBu", vmin=0.0, vmax=0.35)
axes[1].set_title("Convex Quality Optimizer (2020–2026)", fontsize=12, fontweight="bold")
axes[1].set_xticks(range(0, len(cq_sub.columns), 4))
axes[1].set_xticklabels([d.strftime("%Y") for d in cq_sub.columns[::4]], rotation=45)

cbar = fig.colorbar(im1, ax=axes.ravel().tolist(), shrink=0.85, pad=0.02)
cbar.set_label("Target Allocation Weight (Capped at 35%)")

plt.suptitle("Cross-Sectional Weight Transition Heatmap: Uniform vs. Quality-Tilted QP", fontsize=14, fontweight="bold", y=1.02)
plt.show()